In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
# We defined transformation for the training and test sets
# and randomly crops the images for 32x32 pixels and flip the images
# with default probablity of 0.5 so the model can learn invariant orietations
# then it converts the the array into a tensor and scales the pixel values
# from [0,255] to [0.0,1.0]
# then we normalize the pixels of the pictures
# the first tuple is mean and the second is standard deviation
# it extract the mean from the pixels then divided it by standard deviation
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    # Removed transforms.Resize((224, 224)) for the custom Net model
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

transform_test = transforms.Compose([
    # Removed transforms.Resize((224, 224)) for the custom Net model
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

# download the CIFAR10 dataset and apply transfom train to all of the images
trainset = torchvision.datasets.CIFAR10(
    root='./data',
    train=True,
    download=True,
    transform=transform_train
)
# we chose 128 batch size because a bigger batch can cause more efficient
# GPU performance because it works parallel and it also give us more
# stable gradient
# and we use 2 workers, it means subprocesses to speed up data loading
# by performing it in parallel
trainloader = torch.utils.data.DataLoader(
    trainset,
    batch_size=128,
    shuffle=True,
    num_workers=2
)

# download and load the CIFAR-10 test dataset
testset = torchvision.datasets.CIFAR10(
    root='./data',
    train=False,
    download=True,
    transform=transform_test
)

testloader = torch.utils.data.DataLoader(
    testset,
    batch_size=128,
    shuffle=False,
    num_workers=2
)

# define 10 classes
classes = ('plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

In [ ]:
# show image and unormalize
def imshow(img):
    img = img / 2 + 0.5
    npimg = img.numpy()
    plt.imshow(np.transpose(npimg, (1, 2, 0)))
    plt.show()

# random training images
dataiter = iter(trainloader)
images, labels = next(dataiter)

imshow(torchvision.utils.make_grid(images[:4]))
# print labels
print(' '.join(f'{classes[labels[j]]:5s}' for j in range(4)))

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
# we defined cnn and it will sort images into 10 categories
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 6, 5)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(6, 16, 5)
        # The input features for fc1 are recalculated for 32x32 input images:
        # After conv1 (kernel 5): (32-5+1) = 28
        # After pool1 (kernel 2, stride 2): 28/2 = 14
        # After conv2 (kernel 5): (14-5+1) = 10
        # After pool2 (kernel 2, stride 2): 10/2 = 5
        # So, the flattened size before fc1 is 16 channels * 5 * 5 = 400
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)
# we defined leaky relu and convert the output of convolutional layer
# into 1 dimensional vector (torch flatten)
    def forward(self, x):
        x = self.pool(F.leaky_relu(self.conv1(x)))
        x = self.pool(F.leaky_relu(self.conv2(x)))
        x = torch.flatten(x, 1)
        x = F.leaky_relu(self.fc1(x))
        x = F.leaky_relu(self.fc2(x))
        x = self.fc3(x)
        return x

net_leakyrelu_sgd = Net()

In [ ]:
import torch.optim as optim
# we ue entropy for classification task it gives a number which determines how wrong
# the network is and we use SGD optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(net_leakyrelu_sgd.parameters(), lr=0.0001, momentum=0.9)

In [ ]:
num_epochs = 10

# Check if CUDA is available and move model to GPU if it is
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
net_leakyrelu_sgd.to(device)
print(f"Net (LeakyReLU+SGD) moved to {device}")

# we train the leakyrelu nueral network
for epoch in range(num_epochs):
    running_loss = 0.0
    for i, data in enumerate(trainloader, 0):
        inputs, labels = data
        inputs, labels = inputs.to(device), labels.to(device) # Move data to GPU

        optimizer.zero_grad()

        outputs = net_leakyrelu_sgd(inputs)
        loss = criterion(outputs, labels)  # compares the outputs with the original labels
        #and gives a number how bad was the network on the current batch
        loss.backward() # determine the gradient of loss
        optimizer.step()

        running_loss += loss.item()
        if i % 200 == 199:    # print out every 200th batch to see how is the training going
            print(f'Epoch [{epoch + 1}/{num_epochs}], Batch [{i + 1}/{len(trainloader)}] loss: {running_loss / 200:.3f}')
            running_loss = 0.0

Epoch [1/10], Batch [200/391] loss: 2.303
Epoch [2/10], Batch [200/391] loss: 2.303
Epoch [3/10], Batch [200/391] loss: 2.303
Epoch [4/10], Batch [200/391] loss: 2.303
Epoch [5/10], Batch [200/391] loss: 2.303


In [ ]:
correct = 0
total = 0
# we are not training so we do not need to calculate gradient
# we calculate the outputs by running images through the network
# we chose the class with the highest energy for prediction
with torch.no_grad():
    for data in testloader:
        images, labels = data
        outputs = net_leakyrelu_sgd(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f'Accuracy of the network on the 10000 test images: {100 * correct / total:.2f}%')

In [ ]:
import torchvision.models as models
# we are loading a pre-trained AlexNet model and modifier the classifier for CIFAR-10
alexnet = models.alexnet(weights=models.AlexNet_Weights.DEFAULT)

# the last layer is the 6th which originally outputs 1000 classes but we change it for 10
alexnet.classifier[6] = nn.Linear(alexnet.classifier[6].in_features, 10)

# freeze every parameters except the final layer
for param in alexnet.parameters():
    param.requires_grad = False
# unfreeze the parameters in the new final layer for fine-tuning
for param in alexnet.classifier[6].parameters():
    param.requires_grad = True


In [ ]:
# we defined criterion and optimizer for AlexNet fine-tuning
criterion_alexnet = nn.CrossEntropyLoss()
optimizer_alexnet = optim.Adam(alexnet.parameters(), lr=0.0001)

# check if CUDA is available and move model to GPU if it is
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
alexnet.to(device)
print(f"AlexNet moved to {device}")

# we worked with less epoch so it will execute faster and moved the data to GPU
num_epochs_alexnet = 5
print("Starting AlexNet fine-tuning on CIFAR-10...")
for epoch in range(num_epochs_alexnet):
    running_loss = 0.0
    for i, data in enumerate(trainloader, 0):
        inputs, labels = data
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer_alexnet.zero_grad()

        outputs = alexnet(inputs)
        loss = criterion_alexnet(outputs, labels)
        loss.backward()
        optimizer_alexnet.step()

        running_loss += loss.item()
        if i % 200 == 199:
            print(f'Epoch [{epoch + 1}/{num_epochs_alexnet}], Batch [{i + 1}/{len(trainloader)}] loss: {running_loss / 200:.3f}')
            running_loss = 0.0


In [ ]:
correct_alexnet = 0
total_alexnet = 0

print("Evaluating AlexNet fine-tuned model...")
with torch.no_grad():
    for data in testloader:
        images, labels = data
        images, labels = images.to(device), labels.to(device)
        outputs = alexnet(images)
        _, predicted = torch.max(outputs.data, 1)
        total_alexnet += labels.size(0)
        correct_alexnet += (predicted == labels).sum().item()

print(f'Accuracy of AlexNet fine-tuned on CIFAR-10 (10000 test images): {100 * correct_alexnet / total_alexnet:.2f}%')


In [ ]:
transform_mnist = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

mnist_trainset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform_mnist)
mnist_trainloader = torch.utils.data.DataLoader(mnist_trainset, batch_size=64, shuffle=True)

mnist_testset = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform_mnist)
mnist_testloader = torch.utils.data.DataLoader(mnist_testset, batch_size=64, shuffle=False)

print("MNIST dataset prepared.")

In [ ]:
# Simple CNN for MNIST (1 input channel, 10 output classes)
class NetMNIST(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 10, kernel_size=5)
        self.pool = nn.MaxPool2d(2)
        self.conv2 = nn.Conv2d(10, 20, kernel_size=5)
        self.fc = nn.Linear(320, 10) # Adjust input features based on image size after conv/pool

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = torch.flatten(x, 1)
        x = F.relu(self.fc(x))
        return x

net_mnist = NetMNIST()
mnist_device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
net_mnist.to(mnist_device)

criterion_mnist = nn.CrossEntropyLoss()
optimizer_mnist = optim.SGD(net_mnist.parameters(), lr=0.01, momentum=0.5)

num_epochs_mnist = 5 # Reduced epochs for faster execution

print("Starting CNN training on MNIST...")
for epoch in range(num_epochs_mnist):
    running_loss = 0.0
    for batch_idx, (data, target) in enumerate(mnist_trainloader):
        data, target = data.to(mnist_device), target.to(mnist_device)
        optimizer_mnist.zero_grad()
        output = net_mnist(data)
        loss = criterion_mnist(output, target)
        loss.backward()
        optimizer_mnist.step()
        running_loss += loss.item()
        if batch_idx % 100 == 99:
            print(f'Epoch [{epoch + 1}/{num_epochs_mnist}], Batch [{batch_idx + 1}/{len(mnist_trainloader)}] loss: {running_loss / 100:.3f}')
            running_loss = 0.0
print('Finished Training on MNIST!')

In [ ]:
correct_mnist = 0
total_mnist = 0

print("Evaluating CNN on MNIST test set...")
with torch.no_grad():
    for data, target in mnist_testloader:
        data, target = data.to(mnist_device), target.to(mnist_device)
        output = net_mnist(data)
        _, predicted = torch.max(output.data, 1)
        total_mnist += target.size(0)
        correct_mnist += (predicted == target).sum().item()

print(f'Accuracy of CNN on MNIST (10000 test images): {100 * correct_mnist / total_mnist:.2f}%')


In [ ]:
# SVHN dataset has 3 color channels, so we need a different transform and adjust the model
# First, let's define a transform that converts grayscale MNIST images to 3 channels for compatibility with SVHN model.

# Note: SVHN images are 32x32. MNIST images are 28x28. We'll need to resize MNIST for AlexNet later,
# but for a custom CNN we can adjust the architecture.

# For this task, we will train our NetMNIST model on MNIST and then use it as a base for SVHN.
# However, SVHN images are 3 channels and MNIST are 1 channel.
# To make the MNIST-trained model compatible with SVHN, we need to adapt it.
# One common approach is to replicate the single MNIST channel 3 times.

# Let's create a new transform for SVHN that includes resizing to 28x28 if needed for the MNIST model, or we adapt the model.
# Given SVHN images are 32x32, we will create a new model that can handle 32x32 3-channel input.
# And then try to transfer weights of the convolutional layers from the MNIST model.

# SVHN Transform (adjusting to 28x28 for direct compatibility with the existing MNIST-trained CNN structure if possible, or adapt model)
# For simplicity, we will assume we can resize SVHN to 28x28 and adapt the first conv layer

transform_svhn = transforms.Compose([
    transforms.Resize((28, 28)), # Resize SVHN to match MNIST image size
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)) # SVHN normalization
])

svhn_trainset = torchvision.datasets.SVHN(root='./data', split='train', download=True, transform=transform_svhn)
svhn_trainloader = torch.utils.data.DataLoader(svhn_trainset, batch_size=64, shuffle=True)

svhn_testset = torchvision.datasets.SVHN(root='./data', split='test', download=True, transform=transform_svhn)
svhn_testloader = torch.utils.data.DataLoader(svhn_testset, batch_size=64, shuffle=False)

print("SVHN dataset prepared.")

In [ ]:
# Adapt NetMNIST for SVHN (3 channels input)
# Create a new CNN architecture that expects 3 input channels.
# We will copy the weights from the trained MNIST model's convolutional layers.

class NetSVHN(nn.Module):
    def __init__(self, original_mnist_model):
        super().__init__()
        # Modify the first convolutional layer to accept 3 input channels
        # A common trick is to average or repeat the weights from the 1-channel input layer
        self.conv1 = nn.Conv2d(3, 10, kernel_size=5)

        # Initialize new conv1 weights from MNIST conv1 weights
        # We can either sum, average, or just take one channel's weights and repeat
        # For simplicity, let's replicate the single channel weights across 3 input channels
        self.conv1.weight.data = original_mnist_model.conv1.weight.data.repeat(1, 3, 1, 1)
        # Optionally, divide by 3 to keep the magnitude similar
        # self.conv1.weight.data = original_mnist_model.conv1.weight.data.repeat(1, 3, 1, 1) / 3

        self.pool = original_mnist_model.pool
        self.conv2 = original_mnist_model.conv2
        self.fc = original_mnist_model.fc

        # Freeze all layers except the final fully connected layer
        for param in self.parameters():
            param.requires_grad = False

        # Unfreeze the final fully connected layer for fine-tuning on SVHN
        self.fc.weight.requires_grad = True
        self.fc.bias.requires_grad = True

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = torch.flatten(x, 1)
        x = F.relu(self.fc(x))
        return x

# Instantiate the adapted model
net_svhn = NetSVHN(net_mnist) # Pass the trained MNIST model
svhn_device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
net_svhn.to(svhn_device)

criterion_svhn = nn.CrossEntropyLoss()
optimizer_svhn = optim.SGD(filter(lambda p: p.requires_grad, net_svhn.parameters()), lr=0.01, momentum=0.5)

num_epochs_svhn = 5 # Reduced epochs for faster execution

print("Starting Transfer Learning (MNIST -> SVHN)...")
for epoch in range(num_epochs_svhn):
    running_loss = 0.0
    for batch_idx, (data, target) in enumerate(svhn_trainloader):
        data, target = data.to(svhn_device), target.to(svhn_device)
        optimizer_svhn.zero_grad()
        output = net_svhn(data)
        loss = criterion_svhn(output, target)
        loss.backward()
        optimizer_svhn.step()
        running_loss += loss.item()
        if batch_idx % 100 == 99:
            print(f'Epoch [{epoch + 1}/{num_epochs_svhn}], Batch [{batch_idx + 1}/{len(svhn_trainloader)}] loss: {running_loss / 100:.3f}')
            running_loss = 0.0
print('Finished Transfer Learning on SVHN!')

In [ ]:
correct_svhn = 0
total_svhn = 0

print("Evaluating transferred CNN on SVHN test set...")
with torch.no_grad():
    for data, target in svhn_testloader:
        data, target = data.to(svhn_device), target.to(svhn_device)
        output = net_svhn(data)
        _, predicted = torch.max(output.data, 1)
        total_svhn += target.size(0)
        correct_svhn += (predicted == target).sum().item()

print(f'Accuracy of Transferred CNN on SVHN (test images): {100 * correct_svhn / total_svhn:.2f}%')
